<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_05_Real_Provenance_TFL_Readiness_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from urllib.request import Request, urlopen
from html.parser import HTMLParser
import ast, hashlib, json, os, platform, re, subprocess, sys, urllib.error
import pandas as pd

NOW=datetime.now(timezone.utc).isoformat()
COLAB=Path('/content').is_dir()
ROOT=(Path('/content') if COLAB else Path.cwd())/'ICHI2027_phase05'
ROOT.mkdir(parents=True,exist_ok=True)
DRIVE_ROOT=Path('/content/drive/MyDrive/NeuroFHIR_ICHI2027')
DRIVE_OK=False
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_OK=(Path('/content/drive/MyDrive')).is_dir()
SHARE=(DRIVE_ROOT/'phase05_outputs' if DRIVE_OK else ROOT/'phase05_outputs')
SHARE.mkdir(parents=True,exist_ok=True)
PRIVATE=(DRIVE_ROOT/'_private_reproducibility' if DRIVE_OK else ROOT/'private_runtime_only')
PRIVATE.mkdir(parents=True,exist_ok=True)
OFFLINE_TEST=os.environ.get('ICHI2027_OFFLINE_TEST')=='1'
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
print(json.dumps({'phase':'05_real_provenance_and_tfl_readiness',
                  'run_utc':NOW,'python':sys.version.split()[0],
                  'pandas':pd.__version__,'drive_connected':DRIVE_OK},indent=2))


Mounted at /content/drive
{
  "phase": "05_real_provenance_and_tfl_readiness",
  "run_utc": "2026-09-26T22:58:33.939730+00:00",
  "python": "3.13.15",
  "pandas": "2.2.3",
  "drive_connected": true
}


In [2]:
PHASE04_PATHS=[DRIVE_ROOT/'phase04_outputs'/'ICHI2027_Phase04_SHAREABLE.json',
               Path('/content/ICHI2027_Phase04_SHAREABLE.json'),
               Path('/mnt/data/ICHI2027_Phase04_SHAREABLE.json'),
               Path('ICHI2027_Phase04_SHAREABLE.json')]
phase04_path=next((p for p in PHASE04_PATHS if p.is_file()),None)
if phase04_path is None and COLAB:
    from google.colab import files
    uploaded=files.upload()
    name='ICHI2027_Phase04_SHAREABLE.json'
    if name not in uploaded:raise FileNotFoundError('Required real Phase04 audit JSON not supplied.')
    phase04_path=ROOT/name
    phase04_path.write_bytes(uploaded[name]);del uploaded
if phase04_path is None:raise FileNotFoundError('Required real Phase04 audit JSON not found.')
PHASE04=json.loads(phase04_path.read_text(encoding='utf-8'))
assert PHASE04['phase']=='04_real_fhir_serialization_preflight'
assert PHASE04['official_r4_validation']['status']=='PASS'
assert PHASE04['official_r4_validation']['all_generated_resources_validated'] is True
assert PHASE04['experiment_gates']['real_source_fhir_serialization_completed'] is True
assert PHASE04['experiment_gates']['original_tfl_v15_executed'] is False
PREVIOUS_SHA=file_sha(phase04_path)
print(json.dumps({'prerequisite':'PASSED','phase04_validation':PHASE04['official_r4_validation']['status'],
    'source_bundle_count':PHASE04['real_fhir_in_memory']['bundle_count'],
    'source_resource_type_counts':PHASE04['real_fhir_in_memory']['resource_type_counts'],
    'phase04_sha256':PREVIOUS_SHA},indent=2))


{
  "prerequisite": "PASSED",
  "phase04_validation": "PASS",
  "source_bundle_count": 258,
  "source_resource_type_counts": {
    "Patient": 258,
    "ImagingStudy": 516,
    "Observation": 4128
  },
  "phase04_sha256": "451d258c0d0c10b509025c75a30c822599929b013597282677b46dedfbad71d7"
}


In [3]:
PINNED_REPOS={
  'ohdsi_tfl':{'url':'https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',
      'commit':PHASE04['pinned_repositories']['ohdsi_tfl']['commit']},
  'imagewg':{'url':'https://github.com/OHDSI/ImageWG.git',
      'commit':PHASE04['pinned_repositories']['imagewg']['commit']},
  'pyomop':{'url':'https://github.com/dermatologist/pyomop.git',
      'commit':'dfaa6d4062dbc600f52f5377019a347c4caa2cc7'}
}
def fetch_pin(name,spec):
    path=ROOT/'upstream'/name
    path.parent.mkdir(parents=True,exist_ok=True)
    info={'status':'NOT_AVAILABLE','expected_commit':spec['commit'],'actual_commit':None}
    if OFFLINE_TEST:return info,path
    try:
        if not (path/'.git').is_dir():
            subprocess.run(['git','init','-q',str(path)],check=True,timeout=30,capture_output=True)
            subprocess.run(['git','-C',str(path),'remote','add','origin',spec['url']],check=True,timeout=30,capture_output=True)
        subprocess.run(['git','-C',str(path),'fetch','-q','--depth','1','origin',spec['commit']],
                       check=True,timeout=180,capture_output=True)
        subprocess.run(['git','-C',str(path),'checkout','-q','--detach','FETCH_HEAD'],
                       check=True,timeout=30,capture_output=True)
        got=subprocess.run(['git','-C',str(path),'rev-parse','HEAD'],
                check=True,timeout=15,capture_output=True,text=True).stdout.strip()
        info={'status':'VERIFIED' if got==spec['commit'] else 'MISMATCH',
              'expected_commit':spec['commit'],'actual_commit':got}
    except (subprocess.CalledProcessError,subprocess.TimeoutExpired,OSError) as exc:
        info['status']='NETWORK_OR_GIT_ERROR'
        info['error_type']=type(exc).__name__
    return info,path
REPOS={};REPO_PATHS={}
for name,spec in PINNED_REPOS.items():
    REPOS[name],REPO_PATHS[name]=fetch_pin(name,spec)
print(json.dumps(REPOS,indent=2))


{
  "ohdsi_tfl": {
    "status": "VERIFIED",
    "expected_commit": "a4b7265adabf096e84b957de068e24a5ab853425",
    "actual_commit": "a4b7265adabf096e84b957de068e24a5ab853425"
  },
  "imagewg": {
    "status": "VERIFIED",
    "expected_commit": "8f0a19bb94e941bb335a097bc9851fab672621c7",
    "actual_commit": "8f0a19bb94e941bb335a097bc9851fab672621c7"
  },
  "pyomop": {
    "status": "VERIFIED",
    "expected_commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7",
    "actual_commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7"
  }
}


In [4]:
RUNNER='FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb'
EXPECTED_RUNNER_SHA=next((x['sha256'] for x in PHASE04['original_tfl_v15_static_inventory']['candidates']
    if x['path']==RUNNER),None)
assert EXPECTED_RUNNER_SHA,'Phase04 did not inventory the original v15 notebook.'
def code_info(code):
    cleaned=[]
    for raw in code.splitlines():
        stripped=raw.lstrip()
        if stripped.startswith(('!','%','?')): continue
        cleaned.append(raw)
    text='\n'.join(cleaned)
    try:
        tree=ast.parse(text)
    except SyntaxError:
        return {'parsed':False,'function_names':[],'class_names':[], 'imports':[]}
    functions=sorted({n.name for n in ast.walk(tree) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef))})
    classes=sorted({n.name for n in ast.walk(tree) if isinstance(n,ast.ClassDef)})
    imports=sorted({n.module.split('.')[0] for n in ast.walk(tree) if isinstance(n,ast.ImportFrom) and n.module}
                   |{alias.name.split('.')[0] for n in ast.walk(tree) if isinstance(n,ast.Import) for alias in n.names})
    return {'parsed':True,'function_names':functions,'class_names':classes,'imports':imports}
RUNNER_AUDIT={'status':'NOT_INSPECTED','sha256_expected':EXPECTED_RUNNER_SHA,
              'sha256_actual':None,'executed':False,'notebook_cells':None,
              'code_cells':None,'parseable_python_cells':None,'function_names':[],
              'imports':[],'cells_requiring_ipython_shell':None,'potential_local_file_references':[]}
if REPOS['ohdsi_tfl']['status']=='VERIFIED':
    f=REPO_PATHS['ohdsi_tfl']/RUNNER
    if f.is_file():
        actual=file_sha(f)
        RUNNER_AUDIT['sha256_actual']=actual
        RUNNER_AUDIT['status']='HASH_VERIFIED' if actual==EXPECTED_RUNNER_SHA else 'HASH_MISMATCH_STOP'
        if actual!=EXPECTED_RUNNER_SHA:raise RuntimeError('Frozen v15 notebook checksum differs from Phase04.')
        notebook=json.loads(f.read_text(encoding='utf-8'))
        parts=[c for c in notebook['cells'] if c['cell_type']=='code']
        sources=[''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in parts]
        infos=[code_info(s) for s in sources]
        refs=sorted({x for s in sources for x in re.findall(r'(?i)[\w./-]+\.(?:csv|parquet|json|duckdb|db|sqlite|ndjson)',s)
          if '/content/' not in x and 'MyDrive' not in x and 'http' not in x})
        RUNNER_AUDIT.update(notebook_cells=len(notebook['cells']),code_cells=len(parts),
            parseable_python_cells=sum(x['parsed'] for x in infos),
            function_names=sorted({n for x in infos for n in x['function_names']}),
            imports=sorted({n for x in infos for n in x['imports']}),
            cells_requiring_ipython_shell=sum(any(line.lstrip().startswith(('!','%')) for line in s.splitlines()) for s in sources),
            potential_local_file_references=refs[:80])
print(json.dumps(RUNNER_AUDIT,indent=2))


{
  "status": "HASH_VERIFIED",
  "sha256_expected": "5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19",
  "sha256_actual": "5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19",
  "executed": false,
  "notebook_cells": 36,
  "code_cells": 17,
  "parseable_python_cells": 17,
  "function_names": [
    "compute_metrics",
    "join_warning_tokens",
    "nonempty_warning",
    "sha256_file",
    "split_warning_tokens",
    "warning_count"
  ],
  "imports": [
    "collections",
    "datetime",
    "google",
    "hashlib",
    "json",
    "matplotlib",
    "numpy",
    "os",
    "pandas",
    "pathlib",
    "platform",
    "shutil",
    "sys",
    "zipfile"
  ],
  "cells_requiring_ipython_shell": 0,
  "potential_local_file_references": [
    "_fidelity_audit_v10.parquet",
    "_medication_attribution_audit_v14.parquet",
    "_tfl_final_v15.parquet",
    "controlled_source_level_warning_counts_v15.csv",
    "environment_v15.json",
    "final_event_warning_counts_v1

In [5]:
FROZEN_TFL_AUDIT={'status':'NOT_INSPECTED','executed':False}
if REPOS['ohdsi_tfl']['status']=='VERIFIED':
    base=REPO_PATHS['ohdsi_tfl']
    matched=[]
    patterns={name:re.compile(r'\b'+name+r'\b',re.I) for name in
        ('PRESERVED','NORMALIZED','COLLAPSED','AMBIGUOUS','UNMAPPED','DROPPED','TRACEABILITY_LOSS')}
    for f in sorted(base.rglob('*')):
        if not f.is_file() or '.git' in f.parts or f.suffix.lower() not in ('.py','.ipynb','.sql'):continue
        if f.stat().st_size>2500000:continue
        raw=f.read_text(encoding='utf-8',errors='replace')
        terms={key:len(p.findall(raw)) for key,p in patterns.items()}
        if not any(terms.values()):continue
        matched.append({'path':f.relative_to(base).as_posix(),'sha256':file_sha(f),
                        'taxonomy_term_mentions':terms})
    FROZEN_TFL_AUDIT={'status':'STATIC_INSPECTION_ONLY','executed':False,
                      'matching_files':matched,'file_count':len(matched)}
print(json.dumps({'status':FROZEN_TFL_AUDIT['status'],
                  'candidate_files':FROZEN_TFL_AUDIT.get('file_count',0),
                  'original_tfl_executed':False},indent=2))


{
  "status": "STATIC_INSPECTION_ONLY",
  "candidate_files": 13,
  "original_tfl_executed": false
}


In [6]:
class TextFromHTML(HTMLParser):
    def __init__(self):super().__init__();self.text=[];self.skip=0
    def handle_starttag(self,tag,attrs):
        if tag in ('script','style'):self.skip+=1
    def handle_endtag(self,tag):
        if tag in ('script','style') and self.skip:self.skip-=1
    def handle_data(self,data):
        if not self.skip:self.text.append(data)
PUBLIC_DOCS={
    'adni_mri_tables':'https://adni.loni.usc.edu/quick-start-guide-asset/MRI_tables.html',
    'ohdsi_imagewg_readme':'https://raw.githubusercontent.com/OHDSI/ImageWG/8f0a19bb94e941bb335a097bc9851fab672621c7/README.md'}
DOC_AUDIT={}
for name,url in PUBLIC_DOCS.items():
    info={'url':url,'status':'NOT_RETRIEVED','sha256':None,
          'mentions_freesurfer':None,'mentions_ucsf_longitudinal':None,'mentions_volume_unit_mm3':None,
          'verifies_units_for_local_csv':False}
    if not OFFLINE_TEST:
        try:
            req=Request(url,headers={'User-Agent':'ICHI2027-evidence-audit/1.0'})
            with urlopen(req,timeout=30) as res:body=res.read(3_000_001)
            if len(body)>3_000_000:raise ValueError('Unexpectedly large source page.')
            raw=body.decode('utf-8',errors='replace')
            if name=='adni_mri_tables':
                parser=TextFromHTML();parser.feed(raw);text=' '.join(parser.text)
            else:text=raw
            low=text.lower()
            info.update(status='RETRIEVED_REFERENCE_ONLY',sha256=hashlib.sha256(body).hexdigest(),
                        mentions_freesurfer=('freesurfer' in low),
                        mentions_ucsf_longitudinal=('ucsffsl' in low or 'longitudinal freesurfer' in low),
                        mentions_volume_unit_mm3=bool(re.search(r'\bmm\s*(?:\^3|³|3)\b|cubic millimet',low)))
        except (OSError,ValueError,UnicodeError) as exc:
            info.update(status='NETWORK_OR_PARSING_ERROR',error_type=type(exc).__name__)
    DOC_AUDIT[name]=info
print(json.dumps(DOC_AUDIT,indent=2))


{
  "adni_mri_tables": {
    "url": "https://adni.loni.usc.edu/quick-start-guide-asset/MRI_tables.html",
    "status": "NETWORK_OR_PARSING_ERROR",
    "sha256": null,
    "mentions_freesurfer": null,
    "mentions_ucsf_longitudinal": null,
    "mentions_volume_unit_mm3": null,
    "verifies_units_for_local_csv": false,
    "error_type": "ValueError"
  },
  "ohdsi_imagewg_readme": {
    "url": "https://raw.githubusercontent.com/OHDSI/ImageWG/8f0a19bb94e941bb335a097bc9851fab672621c7/README.md",
    "status": "RETRIEVED_REFERENCE_ONLY",
    "sha256": "cf308882e5616d408d76da1dcf535699df3463414bdaca8e05fc5b70032770c6",
    "mentions_freesurfer": false,
    "mentions_ucsf_longitudinal": false,
    "mentions_volume_unit_mm3": false,
    "verifies_units_for_local_csv": false
  }
}


In [7]:
LONG_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
SOURCE_DIRS=[Path('/content'),ROOT/'private_runtime_only',
    DRIVE_ROOT/'private_inputs',DRIVE_ROOT/'inputs',
    Path('/content/ICHI2027_phase04/private_runtime_only'),
    Path('/content/ICHI2027_phase03/private_runtime_only')]
paths=[p/LONG_NAME for p in SOURCE_DIRS if (p/LONG_NAME).is_file()]
if len({file_sha(p) for p in paths})>1:raise RuntimeError('Conflicting longitudinal CSV copies. Stop and resolve.')
source=paths[0] if paths else None
if source is None and COLAB:
    from google.colab import files
    uploaded=files.upload()
    if LONG_NAME not in uploaded:raise FileNotFoundError('Missing real longitudinal MRI CSV.')
    source=ROOT/'private_runtime_only'/LONG_NAME
    source.parent.mkdir(parents=True,exist_ok=True)
    source.write_bytes(uploaded[LONG_NAME]);del uploaded
SOURCE_AUDIT={'status':'NOT_AVAILABLE','source_rows':None,'original_version_columns_present':False}
if source is not None:
    df=pd.read_csv(source,dtype=str,keep_default_na=False,low_memory=False,encoding='utf-8-sig')
    expect=PHASE04['source_profile_check']['longitudinal']
    assert len(df)==expect['rows'] and len(df.columns)==expect['columns']
    required={'RID','BL_VERSION','M12_VERSION','BL_IMAGEUID','M12_IMAGEUID',
              'BL_LONIUID','M12_LONIUID','BL_EXAMDATE','M12_EXAMDATE',
              'BL_LHIPPOC','BL_RHIPPOC','M12_LHIPPOC','M12_RHIPPOC'}
    assert required<=set(df.columns),'Missing columns needed for source provenance audit.'
    assert df['RID'].str.strip().is_unique,'Ambiguous real longitudinal participant identifier.'
    safe={}
    for field in ('BL_VERSION','M12_VERSION','BL_OVERALLQC','M12_OVERALLQC',
                  'BL_STATUS','M12_STATUS'):
        if field not in df:continue
        values=df[field].str.strip()
        counts=values[values.ne('')].value_counts(dropna=False)
        safe[field]={'present_rows':int(values.ne('').sum()),'distinct_values':int(len(counts)),
                     'value_counts_suppressed':True,
                     'freesurfer_keyword_rows':int(values.str.contains(r'freesurfer',case=False,regex=True).sum()) if field.endswith('VERSION') else None}
    SOURCE_AUDIT={'status':'REAL_CSV_READ','source_rows':int(len(df)),
            'source_columns_count':int(len(df.columns)),
            'original_version_columns_present':all(x in df.columns for x in ('BL_VERSION','M12_VERSION')),
            'source_field_completeness':safe,
            'source_unit_column_exists':any('unit' in x.lower() for x in df.columns),
            'raw_dicom_study_uid_confirmed':False,
            'extraction_software_identity_confirmed':False,
            'private_source_file_sha256_saved_to_shareable':False}
    del df
print(json.dumps(SOURCE_AUDIT,indent=2))


{
  "status": "REAL_CSV_READ",
  "source_rows": 258,
  "source_columns_count": 51,
  "original_version_columns_present": true,
  "source_field_completeness": {
    "BL_VERSION": {
      "present_rows": 258,
      "distinct_values": 24,
      "value_counts_suppressed": true,
      "freesurfer_keyword_rows": 0
    },
    "M12_VERSION": {
      "present_rows": 258,
      "distinct_values": 26,
      "value_counts_suppressed": true,
      "freesurfer_keyword_rows": 0
    },
    "BL_OVERALLQC": {
      "present_rows": 258,
      "distinct_values": 2,
      "value_counts_suppressed": true,
      "freesurfer_keyword_rows": null
    },
    "M12_OVERALLQC": {
      "present_rows": 258,
      "distinct_values": 2,
      "value_counts_suppressed": true,
      "freesurfer_keyword_rows": null
    },
    "BL_STATUS": {
      "present_rows": 258,
      "distinct_values": 1,
      "value_counts_suppressed": true,
      "freesurfer_keyword_rows": null
    },
    "M12_STATUS": {
      "present_rows": 25

In [8]:
EVIDENCE_FOLDER=DRIVE_ROOT/'_private_reproducibility'/'provenance'
CANDIDATE_EXTENSIONS={'.ipynb','.py','.csv','.tsv','.json','.txt','.pdf'}
EVIDENCE_AUDIT={'folder_exists':EVIDENCE_FOLDER.is_dir(),
                'candidate_document_count':0,'candidate_types':{},
                'original_trace_state_file_confirmed':False,
                'original_adni_source_dictionary_confirmed':False,
                'verified_column_level_source_mapping':False}
if EVIDENCE_FOLDER.is_dir():
    docs=[p for p in EVIDENCE_FOLDER.rglob('*') if p.is_file() and p.suffix.lower() in CANDIDATE_EXTENSIONS]
    EVIDENCE_AUDIT['candidate_document_count']=len(docs)
    EVIDENCE_AUDIT['candidate_types']=dict(Counter(p.suffix.lower() for p in docs))
    EVIDENCE_AUDIT['files_reviewed_by_human']=False
print(json.dumps(EVIDENCE_AUDIT,indent=2))


{
  "folder_exists": false,
  "candidate_document_count": 0,
  "candidate_types": {},
  "original_trace_state_file_confirmed": false,
  "original_adni_source_dictionary_confirmed": false,
  "verified_column_level_source_mapping": false
}


In [9]:
CSV_REAUDIT={'status':'NOT_AVAILABLE','required_fields_by_table':{},
             'number_of_fields_by_table':{},'image_feature_has_numeric_value_field':None,
             'executable_ddl_found':False,'omop_vocabulary_snapshot_pinned':False}
FIELD_PATH='inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
if REPOS['imagewg']['status']=='VERIFIED':
    field_file=REPO_PATHS['imagewg']/FIELD_PATH
    if field_file.is_file():
        expected=PHASE04['mi_cdm_pinned_field_spec']['field_spec_sha256']
        got=file_sha(field_file)
        if expected!=got:raise RuntimeError('Pinned MI-CDM field spec changed relative to Phase04.')
        import csv
        with field_file.open(encoding='utf-8-sig',newline='') as h:
            rdr=csv.DictReader(h);rows=list(rdr);names={re.sub(r'[^a-z]','',k.lower()):k for k in rdr.fieldnames}
        def fld(row,key):return str(row.get(names[key],'')).strip() if key in names else ''
        bytable={t:[r for r in rows if fld(r,'cdmtablename').upper()==t]
                 for t in ('IMAGE_OCCURRENCE','IMAGE_FEATURE')}
        def required(rows):return [fld(r,'cdmfieldname') for r in rows
              if fld(r,'isrequired').lower() in ('true','1','yes','y','required','x')]
        feature_fields=[fld(r,'cdmfieldname') for r in bytable['IMAGE_FEATURE']]
        CSV_REAUDIT.update(status='VERIFIED_PINNED_FIELD_SPEC',
          field_spec_sha256=got,
          required_fields_by_table={k:required(v) for k,v in bytable.items()},
          number_of_fields_by_table={k:len(v) for k,v in bytable.items()},
          image_feature_has_numeric_value_field=any(x.lower() in ('value_as_number','value','value_num','measurement_value') for x in feature_fields))
print(json.dumps(CSV_REAUDIT,indent=2))


{
  "status": "VERIFIED_PINNED_FIELD_SPEC",
  "required_fields_by_table": {
    "IMAGE_OCCURRENCE": [
      "image_occurrence_id",
      "person_id",
      "procedure_occurrence_id",
      "image_occurrence_date",
      "image_study_UID",
      "image_series_UID",
      "modality_concept_id"
    ],
    "IMAGE_FEATURE": [
      "image_feature_id",
      "person_id",
      "image_occurrence_id",
      "image_feature_concept_id",
      "image_feature_type_concept_id"
    ]
  },
  "number_of_fields_by_table": {
    "IMAGE_OCCURRENCE": 11,
    "IMAGE_FEATURE": 12
  },
  "image_feature_has_numeric_value_field": false,
  "executable_ddl_found": false,
  "omop_vocabulary_snapshot_pinned": false,
  "field_spec_sha256": "af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4"
}


In [10]:
KNOWN_SOURCE={'participant_key':'RID','timepoints':['BL','M12'],
    'source_image_ids':['BL_IMAGEUID','BL_LONIUID','M12_IMAGEUID','M12_LONIUID'],
    'date_columns':['BL_EXAMDATE','M12_EXAMDATE'],
    'available_derived_measurement_events':4128,
    'values_reconstructed_from_wide_data_in_phase03':True,
    'official_fhir_R4_structure_passed_in_phase04':True,
    'measurement_unit':None,'verified_dicom_study_uid':None,
    'verified_dicom_series_uid':None,'verified_software_name':None,
    'original_trace_state':None,'trace_reference_id':None,
    'omop_concept_snapshot':None}
KNOWN_SOURCE['available_derived_measurement_events']=PHASE04['real_fhir_in_memory']['resource_type_counts']['Observation']
MI_REQUIRED=set(CSV_REAUDIT.get('required_fields_by_table',{}).get('IMAGE_OCCURRENCE',[]))
MI_GAPS={
    'procedure_occurrence_id':'No verified source procedure mapping or OMOP procedure target',
    'image_study_UID':'Raw IMAGEUID/LONIUID not verified as DICOM StudyInstanceUID',
    'image_series_UID':'No verified DICOM SeriesInstanceUID',
    'modality_concept_id':'No pinned OMOP vocabulary snapshot and validated MRI concept',
    'image_feature_concept_id':'No pinned OMOP concept for feature',
    'image_feature_type_concept_id':'No pinned OMOP concept for feature type',
    'numeric_measurement_destination':'Pinned IMAGE_FEATURE fields do not establish a validated numeric measurement mapping',
    'quantity_unit':'No authoritative, source-specific unit crosswalk',
    'TRACE_reliability':'Original TRACE state and calibration reference not present in three source CSVs'
}
GATES={
 'phase04_real_fhir_schema_passed':PHASE04['official_r4_validation']['status']=='PASS',
 'frozen_tfl_code_checksum_verified':RUNNER_AUDIT['status']=='HASH_VERIFIED',
 'frozen_tfl_v15_executed':False,
 'pinned_micdm_field_spec_verified':CSV_REAUDIT['status']=='VERIFIED_PINNED_FIELD_SPEC',
 'source_csv_read_this_run':SOURCE_AUDIT['status']=='REAL_CSV_READ',
 'source_units_authoritatively_verified_for_exact_export':False,
 'dicom_study_and_series_uid_verified':False,
 'original_trace_state_and_reference_verified':False,
 'omop_vocabulary_snapshot_pinned':False,
 'executable_micdm_schema_verified':False,
 'ready_for_real_micdm_load':False,
 'real_four_arm_etl_comparison_executed':False
}
REPORT={'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
  'phase':'05_real_provenance_and_tfl_readiness','run_utc':NOW,
  'scientific_results_status':'SOURCE_PROVENANCE_AND_ORIGINAL_METHOD_AUDIT_ONLY_NO_ETL_COMPARISON',
  'privacy':'Aggregate evidence only; no participant IDs, source CSV bytes, source notebook cells, or individual FHIR resources.',
  'environment':{'python':sys.version.split()[0],'pandas':pd.__version__,
                 'platform':platform.platform(),'google_drive_output_enabled':DRIVE_OK},
  'phase04_sha256':PREVIOUS_SHA,
  'phase04_existing_verified_results':{'FHIR_R4_schema_validation':'PASS',
        'actual_bundle_count':PHASE04['real_fhir_in_memory']['bundle_count'],
        'actual_resource_counts':PHASE04['real_fhir_in_memory']['resource_type_counts']},
  'pinned_repositories':REPOS,'frozen_tfl_v15_runner_audit':RUNNER_AUDIT,
  'frozen_tfl_static_taxonomy_audit':FROZEN_TFL_AUDIT,
  'public_reference_documents':DOC_AUDIT,
  'private_original_source_profile':SOURCE_AUDIT,
  'additional_provenance_document_inventory':EVIDENCE_AUDIT,
  'pinned_micdm_field_spec':CSV_REAUDIT,'known_source_contract':KNOWN_SOURCE,
  'unresolved_real_micdm_mapping_requirements':MI_GAPS,
  'experiment_gates':GATES,'actual_four_arm_comparator_scores':None,
  'real_micdm_load_results':None,
  'next_stage_gate':'PROVE_EXACT_ADNI_EXPORT_UNIT_AND_PROCESSING_LINEAGE; AUDIT_AND_EXECUTE_ORIGINAL_TFL_ADAPTER_ON_VALID_INPUT; PIN_VOCAB_AND_TARGET_DDL'}
SHARE_PATH=SHARE/'ICHI2027_Phase05_SHAREABLE.json'
SHARE_PATH.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({'shareable_json_saved':str(SHARE_PATH),
   'verified_tfl_source':GATES['frozen_tfl_code_checksum_verified'],
   'unit_and_provenance_requirements_resolved':GATES['source_units_authoritatively_verified_for_exact_export'],
   'real_omop_load_run':False,'comparator_scores':None},indent=2))


{
  "shareable_json_saved": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase05_outputs/ICHI2027_Phase05_SHAREABLE.json",
  "verified_tfl_source": true,
  "unit_and_provenance_requirements_resolved": false,
  "real_omop_load_run": false,
  "comparator_scores": null
}


In [11]:
if COLAB:
    from google.colab import files
    files.download(str(SHARE_PATH))
else:
    print(str(SHARE_PATH.resolve()))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>